# psychscanner — Feedback Mechanism Tutorial

This notebook shows how to add **trial-level feedback** to a psychscanner experiment using
the `FeedbackBase` API.  The example is built around the n-back task
from `examples/nback_msg_injection.py`, but every step generalises to any task.

## What is feedback in psychscanner?

After each trial the LLM produces a response.  Feedback lets you inspect that response
and inject a short message at the start of the *next* trial.  Typical uses:

- Tell the model whether its last answer was correct.
- Remind it of task rules it violated.
- Give a hint without revealing the answer outright.

`psychscanner` supports this through the `FeedbackBase` abstract class, which
`TaskRunner` calls automatically between trials when `card.feedback = True`.

In [1]:
# ── Setup: ensure examples/ is on the path ──────────────────────────────────
import sys, os

# Locate the examples/ folder regardless of where Jupyter was launched from.
# The notebook itself lives in examples/, so we look for nback_msg_injection.py
# starting from cwd and stepping up one level if needed.
_cwd = os.getcwd()
for _candidate in [_cwd, os.path.join(_cwd, 'examples')]:
    if os.path.exists(os.path.join(_candidate, 'nback_msg_injection.py')):
        sys.path.insert(0, _candidate)
        print(f'examples dir: {_candidate}')
        break
else:
    raise RuntimeError(
        'Could not find nback_msg_injection.py. '
        'Open Jupyter from the psychscanner project root or the examples/ folder.'
    )

examples dir: /Users/saurabhext/Documents/Projects/PSYCHSCANNER/psychscanner/examples


---
## 1. The `FeedbackBase` contract

Every feedback handler must subclass `FeedbackBase` and implement one method:

```python
def on_response(self, trial: dict, response: dict) -> str | None:
    ...
```

| Argument | What it contains |
|---|---|
| `trial` | Full trial dict from the task JSON (`trcode`, `stimulus`, `corrAns`, …) |
| `response` | Model response as a plain Python dict (pydantic `.model_dump()` for structured output, or `{"content": raw_string}` for unstructured) |

Return a feedback **string** (usually JSON) or `None` to skip feedback for that trial.

An optional `inject_feedback(input_dict, fb_str)` method controls how the feedback string
is merged into the next trial's input.  The default merges it with the current trial
stimulus into a single `HumanMessage`.

In [2]:
from psychscanner.feedback import FeedbackBase

# Minimal example — always give the same hint
class SimpleFeedback(FeedbackBase):
    def on_response(self, trial, response):
        return 'Remember to follow all task instructions carefully.'

assert issubclass(SimpleFeedback, FeedbackBase)
fb = SimpleFeedback()
print(fb.on_response({}, {}))

/opt/anaconda3/envs/psyscan/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Remember to follow all task instructions carefully.


---
## 2. The n-back feedback handler

The file `examples/nback_msg_injection.py` contains `NbackFeedback`, a complete
feedback handler for the SweetPea-generated n-back task
(`examples/tasks/nback_sweetpea_n2.tcard.psyscan`).

Key design points:
- The task has no encoding/test phase split (unlike the older Reality Monitoring
  example this tutorial used to be built around) — every trial is a single
  "is this a match?" judgment against a precomputed `corrAns`, so there's no
  cross-trial state to track. `NbackFeedback` carries no `__init__` at all.
- `on_response(trial, response)` receives a pre-parsed dict — no manual `eval()` or
  `parser_status` string needed.
- `generate_fb_response` is a standalone pure function, easy to unit-test
  independently of any model call (as below).

In [3]:
from nback_msg_injection import NbackFeedback, generate_fb_response
import json

# ── A match trial, answered correctly ──────────────────────────────────────
trial_match = {'trcode': 'nback_sweetpea_n2_05', 'corrAns': 'match'}
fb_correct = generate_fb_response(trial_match, 'match')
print('Feedback (correct):')
print(json.dumps(json.loads(fb_correct), indent=2))
print()

# ── Same trial, answered wrong ─────────────────────────────────────────────
fb_wrong = generate_fb_response(trial_match, 'no-match')
print('Feedback (wrong):')
print(json.dumps(json.loads(fb_wrong), indent=2))

Feedback (correct):
{
  "Feedback on previous response": {
    "response feedback": "**CORRECT: 'match' was the right judgment.**"
  }
}

Feedback (wrong):
{
  "Feedback on previous response": {
    "response feedback": "**INCORRECT: You answered 'no-match', but the correct judgment was 'match'.**"
  }
}


In [4]:
# ── An unparseable response — neither "match" nor "no-match" appears ───────
trial_nomatch = {'trcode': 'nback_sweetpea_n2_02', 'corrAns': 'no-match'}
fb_unparsed = generate_fb_response(trial_nomatch, "I think it's the letter C")
print('Feedback (unparseable):')
print(json.dumps(json.loads(fb_unparsed), indent=2))

Feedback (unparseable):
{
  "Feedback on previous response": {
    "response feedback": "**INCORRECT: Could not find 'match' or 'no-match' in your response ('I think it's the letter C'). The correct answer was 'no-match'. Reply with only 'match' or 'no-match'.**"
  }
}


In [5]:
# ── Same logic through the FeedbackBase.on_response contract ───────────────
# TaskRunner calls this, not generate_fb_response directly -- response is a
# plain dict, {'content': '<raw text>'} for this card's unstructured output.
handler = NbackFeedback()
fb = handler.on_response(trial_match, {'content': 'match'})
print(json.loads(fb)['Feedback on previous response']['response feedback'])

**CORRECT: 'match' was the right judgment.**


In [6]:
# ── NbackFeedback is stateless, unlike RM's Stim_Trial_Injection ───────────
# psychscanner still creates one instance per participant (the FeedbackBase
# contract), but NbackFeedback has no __init__ / instance state to isolate --
# every instance behaves identically because there's no cross-trial history
# to track (each trial's corrAns is already precomputed and self-contained).
handler_p1 = NbackFeedback()
handler_p2 = NbackFeedback()

fb_p1 = handler_p1.on_response(trial_match, {'content': 'match'})
fb_p2 = handler_p2.on_response(trial_match, {'content': 'match'})
print('Same result from independent instances:', fb_p1 == fb_p2)

Same result from independent instances: True


---
## 3. Wiring feedback into an experiment

Pass the **class** (not an instance) as `feedback_fn` in your `ExpCard`.  psychscanner
creates one fresh instance per participant via `feedback_fn()`.

```python
from psychscanner import ExpCard, ScannerModel
from nback_msg_injection import NbackFeedback

card_in = ExpCard(
    model='smollm2:360m-instruct-fp16',
    family='ollama',
    memory='Convo',
    task_file='examples/tasks/nback_sweetpea_n2.tcard.psyscan',
    parser='0',                         # free text, graded by substring match
    feedback=True,                      # also accepts '1' for backward compat
    feedback_fn=NbackFeedback,          # class, not an instance
    ...
)

scanner = ScannerModel(expcard=card_in)
results = scanner.run()
```

See [07_nback_feedback_task.ipynb](07_nback_feedback_task.ipynb) for this wired up
end-to-end against a real local model.

---
## 4. Opting individual trials in / out of feedback

Add an `"fb": false` key to any trial in your task JSON to skip feedback for that trial.
Trials without the key default to `true` (feedback on).

```json
{
  "trcode": "practice_01",
  "fb": false,
  "hmsg": { "..." : "..." }
}
```

Useful for:
- Practice / warm-up trials that should not carry feedback into the first real trial.
- Task-instruction acknowledgement trials where `on_response` already returns `None`.
- Any trial where you want a clean prompt slate.

---
## 5. Customising the injection format

The default `inject_feedback` wraps feedback and the current stimulus into one JSON
`HumanMessage`.  Override it to change the format:

In [7]:
import json
from langchain_core.messages import HumanMessage
from psychscanner.feedback import FeedbackBase

class PlainTextFeedback(FeedbackBase):
    """Inject feedback as a plain-text prefix rather than JSON."""

    def on_response(self, trial, response):
        return f"Previous trial feedback: {response.get('content', '')[:40]}"

    def inject_feedback(self, input_dict, fb_str):
        trial_content = input_dict['inputs'][0].content
        combined = f"{fb_str}\n\n---\n{trial_content}"
        input_dict['inputs'] = [HumanMessage(combined)]
        return input_dict

# Demo
fb = PlainTextFeedback()
input_dict = {'inputs': [HumanMessage('Current trial: word pair [apple / __]')], 'system_message': 'sys'}
updated = fb.inject_feedback(input_dict, 'Previous trial feedback: CORRECT')
print(updated['inputs'][0].content)

Previous trial feedback: CORRECT

---
Current trial: word pair [apple / __]


---
## 6. Inspecting saved feedback in results

Every trial output dict now contains an `fb_response` key with the feedback string
generated after *that* trial (and injected before the next one).

In [8]:
# Hypothetical post-run inspection
# results = scanner.run()   # list of per-participant data

# for participant_data in results:
#     for trial_record in participant_data:
#         trcode = trial_record['trcode']
#         fb = trial_record.get('fb_response')   # None when no feedback was generated
#         print(f"{trcode}: feedback generated = {fb is not None}")

print('Access feedback via:  trial_record["fb_response"]')
print('It is None when on_response returned None for that trial.')

Access feedback via:  trial_record["fb_response"]
It is None when on_response returned None for that trial.


---
## 7. Migration from the old API

### Before (old)

```python
# module-level global — breaks between participants!
all_words_in_use = []

class OldFeedback:
    def __init__(self, trial_data):          # re-instantiated every trial
        self.trial_data = trial_data

    def generate_feedback(self, trdata, pred_dict, input_dict, parser_status, trial_item_collector):
        message = pred_dict.content
        if parser_status == '1':
            message = eval(message)          # manually parse
        ...

    def update_trial_stim(self, finput, fb_response):
        ...

card_in = ExpCard(feedback='1', feedback_fn=OldFeedback)
```

### After (new)

```python
from psychscanner.feedback import FeedbackBase

class NewFeedback(FeedbackBase):             # instantiated once per simulation
    def __init__(self):
        self.all_words_in_use = []           # instance variable — safe across participants

    def on_response(self, trial, response):  # response is already a plain dict
        word_2 = response.get('Word_2', '')
        ...
        return feedback_string_or_none

card_in = ExpCard(feedback=True, feedback_fn=NewFeedback)  # True or '1' both work
```

Summary of changes:
1. Subclass `FeedbackBase` instead of plain `object`.
2. Rename `generate_feedback` → `on_response`; signature simplifies to `(self, trial, response)`.
3. Move module-level globals into `self.__init__`.
4. Remove `update_trial_stim` — the base class `inject_feedback` handles injection (or override it).
5. Change `feedback='1'` → `feedback=True` (both still work).

---
## Further reading

Advanced applications of trial-by-trial feedback injection:

1. **["Reflexion: Language Agents with Verbal Reinforcement Learning"](https://arxiv.org/abs/2303.11366)** (Shinn et al., 2023) — agents that improve across trials purely from injected linguistic feedback, the same mechanism `FeedbackBase.on_response` implements here at the scale of a full learning loop.
2. **["Improving Interactive In-Context Learning from Natural Language Feedback"](https://arxiv.org/abs/2602.16066)** (2026) — treats learning from natural-language feedback (like the `fb_response` string this API injects) as a trainable skill rather than an incidental effect.